In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.quantization
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
# -------------------------------
# Dataset utilities for MNIST
# -------------------------------
def get_mnist_datasets(data_dir, batch_size):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = torchvision.datasets.MNIST(
        root=data_dir, train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.MNIST(
        root=data_dir, train=False, download=True, transform=transform
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

# -------------------------------
# Custom 4-bit QConfig for QAT
# -------------------------------

def get_2bit_qconfig():
    activation_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=0,
        quant_max=3,  # 4-bit: 16 levels for activations
        dtype=torch.quint8,
        reduce_range=False
    )
    weight_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=-2,
        quant_max=1,   # 4-bit symmetric range for weights
        dtype=torch.qint8,
        reduce_range=False
    )
    return torch.quantization.QConfig(activation=activation_fake_quant, weight=weight_fake_quant)


def get_4bit_qconfig():
    activation_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=0,
        quant_max=15,  # 4-bit: 16 levels for activations
        dtype=torch.quint8,
        reduce_range=False
    )
    weight_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=-8,
        quant_max=7,   # 4-bit symmetric range for weights
        dtype=torch.qint8,
        reduce_range=False
    )
    return torch.quantization.QConfig(activation=activation_fake_quant, weight=weight_fake_quant)


def get_8bit_qconfig():
    activation_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=0,
        quant_max=255,  # 4-bit: 16 levels for activations
        dtype=torch.quint8,
        reduce_range=False
    )
    weight_fake_quant = torch.quantization.FakeQuantize.with_args(
        observer=torch.quantization.MovingAverageMinMaxObserver,
        quant_min=-128,
        quant_max=127,   # 4-bit symmetric range for weights
        dtype=torch.qint8,
        reduce_range=False
    )
    return torch.quantization.QConfig(activation=activation_fake_quant, weight=weight_fake_quant)


# -------------------------------
# LeNet-5 Model with QAT Stubs
# -------------------------------
class Quad(nn.Module):
    def forward(self, x):
        return x*x

class LeNet_QAT(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet_QAT, self).__init__()
        # Insert quantization stubs
        self.quant = torch.quantization.QuantStub()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, padding=2, stride=2)
        self.bn1 = nn.BatchNorm2d(32)
        self.act1 = nn.ReLU()

        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=2, stride=2)
        self.bn2 = nn.BatchNorm2d(64)
        self.act2 = nn.ReLU()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(7 * 7 * 64, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.act3 = nn.ReLU()

        self.fc2 = nn.Linear(512, num_classes)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        # Quantize the input
        x = self.quant(x)
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.conv2(x)))
        x = self.flatten(x)
        x = self.act3(self.bn3(self.fc1(x)))
        x = self.fc2(x)
        # Dequantize the output
        x = self.dequant(x)
        return x

# -------------------------------
# Training Functions
# -------------------------------
def train(model, train_loader, test_loader, epochs, lr, momentum, weight_decay, save_path=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0

    for epoch in range(epochs):
        train_epoch(epoch, model, train_loader, criterion, optimizer, device)
        test_acc = test_epoch(model, test_loader, criterion, device)

        if save_path and test_acc > best_acc:
            print(f"Saving model with accuracy: {test_acc:.3f}")
            best_acc = test_acc
            state = {
                "model_state_dict": model.state_dict(),
                "acc": test_acc,
                "epoch": epoch,
            }
            torch.save(state, save_path)

        scheduler.step()

def train_epoch(epoch, model, train_loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    print(f"\nEpoch: {epoch + 1}")
    train_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc="Training", leave=False)

    for batch_idx, (inputs, targets) in train_bar:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        train_bar.set_postfix({
            "Loss": f"{train_loss / (batch_idx + 1):.3f}",
            "Acc": f"{100. * correct / total:.3f}% ({correct}/{total})"
        })

def test_epoch(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    test_bar = tqdm(enumerate(test_loader), total=len(test_loader), desc="Testing", leave=False)

    with torch.no_grad():
        for batch_idx, (inputs, targets) in test_bar:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            test_bar.set_postfix({
                "Loss": f"{test_loss / (batch_idx + 1):.3f}",
                "Acc": f"{100. * correct / total:.3f}% ({correct}/{total})"
            })

    acc = 100. * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc

def train_on_mnist_qat(model, data_dir="./data", epochs=10, batch_size=128, lr=0.1,
                       momentum=0.9, weight_decay=5e-4, save_path=None, use_qat=True, bit_width=4):
    """
    Train a model on the MNIST dataset with optional 4-bit Quantization Aware Training.
    """



    if use_qat:
        if bit_width == 2:
            model.qconfig = get_2bit_qconfig()
        elif bit_width == 4:
            model.qconfig = get_4bit_qconfig()
        elif bit_width == 8:
            model.qconfig = get_8bit_qconfig()
        torch.quantization.prepare_qat(model, inplace=True)

    train_loader, test_loader = get_mnist_datasets(data_dir, batch_size)
    train(model, train_loader, test_loader, epochs, lr, momentum, weight_decay, save_path)

    if use_qat:
        model.eval()
        torch.quantization.convert(model, inplace=True)

    return model

# -------------------------------
# Main Entry Point
# -------------------------------
    # Create an instance of the LeNet-5 model with QAT support
model = LeNet_QAT()

# # Train on MNIST with 4-bit QAT. Adjust epochs and hyperparameters as needed.
# trained_model = train_on_mnist_qat(
#     model,
#     data_dir="./data",
#     epochs=3,           # increase this for better performance
#     batch_size=128,
#     lr=0.1,
#     momentum=0.9,
#     weight_decay=5e-4,
#     save_path="lenet5_qat.pth",
#     use_qat=True
# )

# print("Training completed. Model saved as 'lenet5_qat.pth'")

In [ ]:
model = LeNet_QAT()
# Train on MNIST with 4-bit QAT. Adjust epochs and hyperparameters as needed.
trained_model_2bit = train_on_mnist_qat(
    model,
    data_dir="./data",
    epochs=3,           # increase this for better performance
    batch_size=128,
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    save_path="lenet5_qat_2bit.pth",
    use_qat=True,
    bit_width=2)



Epoch: 1


Test Accuracy: 11.18%
Saving model with accuracy: 11.180

Epoch: 2


Test Accuracy: 10.01%

Epoch: 3


Test Accuracy: 9.83%


In [ ]:
model = LeNet_QAT()
# Train on MNIST with 4-bit QAT. Adjust epochs and hyperparameters as needed.
trained_model_4bit = train_on_mnist_qat(
    model,
    data_dir="./data",
    epochs=3,           # increase this for better performance
    batch_size=128,
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    save_path="lenet5_qat_4bit.pth",
    use_qat=True,
    bit_width=4)



Epoch: 1


Test Accuracy: 98.36%
Saving model with accuracy: 98.360

Epoch: 2


Test Accuracy: 98.80%
Saving model with accuracy: 98.800

Epoch: 3


Test Accuracy: 98.87%
Saving model with accuracy: 98.870


In [ ]:
model = LeNet_QAT()
# Train on MNIST with 4-bit QAT. Adjust epochs and hyperparameters as needed.
trained_model_8bit = train_on_mnist_qat(
    model,
    data_dir="./data",
    epochs=3,           # increase this for better performance
    batch_size=128,
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    save_path="lenet5_qat_8bit.pth",
    use_qat=True,
    bit_width=8)


Epoch: 1


Test Accuracy: 98.84%
Saving model with accuracy: 98.840

Epoch: 2


Test Accuracy: 99.21%
Saving model with accuracy: 99.210

Epoch: 3


Test Accuracy: 99.35%
Saving model with accuracy: 99.350
